# 🤖 Assistant RAG IFOAD-UJKZ — Étape 2 : Agent + Interface



## 0. Installation

In [1]:
!pip install -q streamlit chromadb sentence-transformers groq pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/

## 1. Récupération de la base ChromaDB

In [2]:
!unzip -q chroma_db_ifoad.zip -d ./chroma_db_ifoad

In [3]:
import os

CHROMA_DB_PATH = "/content/chroma_db_ifoad"

if os.path.exists(CHROMA_DB_PATH):
    print("✅ Base ChromaDB déjà présente dans", CHROMA_DB_PATH)
else:
    print("⚠️ Base non trouvée. Choisissez une option :")
    print("   Option A : uploadez chroma_db_ifoad.zip et décommentez la cellule ci-dessous.")
    print("   Option B : relancez le notebook 01_ingestion_vectorisation.ipynb d'abord.")

✅ Base ChromaDB déjà présente dans /content/chroma_db_ifoad


## 2. Configuration de la clé API Groq


In [4]:
import os
from getpass import getpass

# Saisie sécurisée (le texte ne s'affiche pas)
groq_key = getpass("🔑 Entrez votre clé API Groq (gsk_...) : ")
os.environ["GROQ_API_KEY"] = groq_key
print("✅ Clé Groq enregistrée dans l'environnement.")

🔑 Entrez votre clé API Groq (gsk_...) : ··········
✅ Clé Groq enregistrée dans l'environnement.


## 3. Test rapide du pipeline RAG avant de lancer l'interface

On vérifie que la connexion ChromaDB → Groq fonctionne correctement avant d'ouvrir l'interface web.

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

COLLECTION_NAME   = "ifoad_communiques"
EMBEDDING_MODEL   = "paraphrase-multilingual-MiniLM-L12-v2"
GROQ_MODEL        = "llama-3.3-70b-versatile"
N_RESULTS         = 4
SIMILARITY_CUTOFF = 0.55

print("⏳ Chargement des ressources...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
client     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = client.get_collection(COLLECTION_NAME)
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

print(f"✅ Base chargée : {collection.count()} segments indexés")

SYSTEM_PROMPT = """Tu es l'Assistant Officiel de l'IFOAD-UJKZ.
Réponds UNIQUEMENT à partir du CONTEXTE fourni, en français.
Si l'information n'est pas dans le contexte, dis-le clairement sans inventer.
Cite toujours la source (formation, type de document) en fin de réponse.

CONTEXTE :
{context}"""

def rag_query(question):
    # 1. Retrieval
    q_vec = embedding_model.encode([question], normalize_embeddings=True)
    results = collection.query(query_embeddings=q_vec.tolist(), n_results=N_RESULTS)
    chunks = [
        results["documents"][0][i]
        for i in range(len(results["ids"][0]))
        if results["distances"][0][i] <= SIMILARITY_CUTOFF
    ]
    context = "\n\n---\n\n".join(chunks)

    if not context:
        return "Je ne dispose pas de cette information dans ma base de connaissances actuelle."

    # 2. Generation
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
            {"role": "user",   "content": question},
        ],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content


# Tests
questions_test = [
    "Quels sont les frais d'inscription pour le Master Data Science ?",
    "Quand est-ce que le projet final doit être rendu ?",
    "Quelles pièces faut-il pour candidater à la Licence Informatique Appliquée ?",
]
for q in questions_test:
    print(f"\n{'='*60}")
    print(f"Q : {q}")
    print(f"R : {rag_query(q)}")

⏳ Chargement des ressources...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Base chargée : 147 segments indexés

Q : Quels sont les frais d'inscription pour le Master Data Science ?
R : Je ne dispose pas de cette information dans ma base de connaissances actuelle.

Q : Quand est-ce que le projet final doit être rendu ?
R : Il y a plusieurs projets finals à rendre, les dates sont les suivantes :
- Le projet final du cours EDD sur les EDW doit être rendu le 15/07/2026 à 00:00.
- Le projet final du cours 1INF1102_1 doit être rendu le 06/07/2026 à 00:00.
- L'examen final du cours 1INF2202, qui inclut un projet individuel sur une semaine, doit être effectué le 09/07/2026 à 23:00.

Source : Calendrier académique IFOAD-UJKZ.

Q : Quelles pièces faut-il pour candidater à la Licence Informatique Appliquée ?
R : Je ne dispose pas de cette information dans ma base de connaissances actuelle.


## 4. Écriture du fichier app.py


In [6]:
app_code = '''
import os
import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq

CHROMA_DB_PATH    = "/content/chroma_db_ifoad"
COLLECTION_NAME   = "ifoad_communiques"
EMBEDDING_MODEL   = "paraphrase-multilingual-MiniLM-L12-v2"
GROQ_MODEL        = "llama-3.3-70b-versatile"
N_RESULTS         = 4
SIMILARITY_CUTOFF = 0.55

SYSTEM_PROMPT = """Tu es l\'Assistant Officiel de l\'IFOAD-UJKZ (Institut de Formation Ouverte et à Distance de l\'Université Joseph Ki-Zerbo), au Burkina Faso.

Ton rôle est de répondre avec précision aux questions des étudiants et candidats concernant :
- Les formations proposées (Master Data Science, Licences, Certificats...)
- Les conditions de candidature et pièces à fournir
- Les frais d\'inscription et de formation
- Les calendriers académiques, dates d\'examens et dépôts de projets
- Les modalités d\'enseignement (en ligne, présentiel, plateforme Moodle...)

RÈGLES IMPORTANTES :
1. Réponds UNIQUEMENT à partir des informations fournies dans le CONTEXTE ci-dessous.
2. Si le contexte ne contient pas l\'information demandée, réponds exactement : "Je ne dispose pas de cette information dans ma base de connaissances actuelle. Je vous invite à contacter directement l\'IFOAD au 63375257 ou à écrire à urbain.traore@ujkz.bf"
3. Ne jamais inventer de chiffres, de dates ou de noms.
4. Cite toujours la source (nom du document et formation concernée) à la fin de ta réponse.
5. Réponds en français, avec un ton professionnel et bienveillant.

CONTEXTE :
{context}"""

@st.cache_resource(show_spinner="Chargement de la base de connaissances...")
def load_resources():
    emb = SentenceTransformer(EMBEDDING_MODEL)
    cli = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    col = cli.get_collection(COLLECTION_NAME)
    return emb, col

@st.cache_resource(show_spinner=False)
def load_groq():
    key = os.environ.get("GROQ_API_KEY", "")
    return Groq(api_key=key) if key else None

def retrieve(query, emb, col):
    vec = emb.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=vec.tolist(), n_results=N_RESULTS)
    sources, parts = [], []
    for i in range(len(res["ids"][0])):
        if res["distances"][0][i] > SIMILARITY_CUTOFF:
            continue
        parts.append(res["documents"][0][i])
        m = res["metadatas"][0][i]
        sources.append({"formation": m.get("formation","-"), "type_doc": m.get("type_doc","-")})
    return "\\n\\n---\\n\\n".join(parts), sources

def answer(question, context, history, groq_cli):
    if not groq_cli:
        return "⚠️ Clé API Groq manquante."
    if not context:
        return "Je ne dispose pas de cette information dans ma base de connaissances actuelle. Contactez l\'IFOAD au **63375257** ou **urbain.traore@ujkz.bf**."
    msgs = [{"role":"system","content":SYSTEM_PROMPT.format(context=context)}]
    for m in history[-8:]:
        msgs.append({"role":m["role"],"content":m["content"]})
    msgs.append({"role":"user","content":question})
    r = groq_cli.chat.completions.create(model=GROQ_MODEL, messages=msgs, temperature=0.2, max_tokens=1024)
    return r.choices[0].message.content

def main():
    st.set_page_config(page_title="Assistant IFOAD-UJKZ", page_icon="🎓", layout="centered")
    st.markdown(\'\'\'<style>
    .header{background:#1B5E20;color:white;padding:1.2rem 1.5rem 0.8rem;border-radius:12px;margin-bottom:1.5rem}
    .header h1{font-size:1.4rem;margin:0;font-weight:700}
    .header p{margin:.2rem 0 0;font-size:.88rem;opacity:.85}
    .cu{background:#1B5E20;color:white;padding:.7rem 1rem;border-radius:18px 18px 4px 18px;margin:.4rem 0 .4rem 15%;font-size:.95rem}
    .ca{background:#E8F5E9;padding:.7rem 1rem;border-radius:18px 18px 18px 4px;margin:.4rem 15% .4rem 0;font-size:.95rem;border-left:3px solid #1B5E20}
    .st{display:inline-block;background:#f0f0f0;color:#555;font-size:.72rem;padding:2px 8px;border-radius:20px;margin:2px 2px 0 0}
    #MainMenu,footer{visibility:hidden}
    </style>\'\'\', unsafe_allow_html=True)
    st.markdown(\'<div class="header"><h1>🎓 Assistant IFOAD-UJKZ</h1><p>Université Joseph Ki-Zerbo · Institut de Formation Ouverte et à Distance</p></div>\', unsafe_allow_html=True)

    try:
        emb, col = load_resources()
        groq_cli = load_groq()
        st.caption(f"📚 Base de connaissances : **{col.count()} segments** indexés")
    except Exception as e:
        st.error(f"❌ Impossible de charger la base ChromaDB : {e}")
        return

    if not groq_cli:
        with st.sidebar:
            st.subheader("🔑 Clé API Groq")
            k = st.text_input("Clé Groq", type="password", placeholder="gsk_...")
            if k:
                os.environ["GROQ_API_KEY"] = k
                st.cache_resource.clear()
                st.rerun()
            st.markdown("[Obtenir une clé gratuite →](https://console.groq.com)")

    if "messages" not in st.session_state:
        st.session_state.messages = [{"role":"assistant","content":"Bonjour ! Je suis l\'assistant de l\'IFOAD-UJKZ. Je peux vous renseigner sur les **formations**, les **conditions de candidature**, les **frais d\'inscription** et les **dates importantes**.\\n\\nComment puis-je vous aider ?","sources":[]}]

    for msg in st.session_state.messages:
        css = "cu" if msg["role"]=="user" else "ca"
        st.markdown(f\'<div class="{css}">{msg["content"]}</div>\', unsafe_allow_html=True)
        if msg.get("sources"):
            for s in msg["sources"]:
                st.markdown(f\'<span class="st">📄 {s["formation"]} — {s["type_doc"]}</span>\', unsafe_allow_html=True)

    q = st.chat_input("Posez votre question sur l\'IFOAD...")
    if q:
        st.session_state.messages.append({"role":"user","content":q})
        st.markdown(f\'<div class="cu">{q}</div>\', unsafe_allow_html=True)
        with st.spinner("Recherche en cours..."):
            ctx, srcs = retrieve(q, emb, col)
            rep = answer(q, ctx, [m for m in st.session_state.messages if m["role"]!="system"], groq_cli)
        st.markdown(f\'<div class="ca">{rep}</div>\', unsafe_allow_html=True)
        for s in srcs:
            st.markdown(f\'<span class="st">📄 {s["formation"]} — {s["type_doc"]}</span>\', unsafe_allow_html=True)
        st.session_state.messages.append({"role":"assistant","content":rep,"sources":srcs})

    with st.sidebar:
        st.subheader("💡 Questions fréquentes")
        for s in ["Frais d\'inscription Master Data Science ?","Conditions de candidature Licence ?","Date limite dépôt projet final ?","Comment candidater sur Campus Faso ?"]:
            if st.button(s, use_container_width=True):
                st.session_state.messages.append({"role":"user","content":s})
                ctx, srcs = retrieve(s, emb, col)
                rep = answer(s, ctx, st.session_state.messages, groq_cli)
                st.session_state.messages.append({"role":"assistant","content":rep,"sources":srcs})
                st.rerun()
        st.divider()
        if st.button("🗑️ Effacer la conversation", use_container_width=True):
            st.session_state.messages = []
            st.rerun()

if __name__ == "__main__":
    main()
'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py écrit dans /content/app.py")

✅ app.py écrit dans /content/app.py


## 5. Lancement de l'application Streamlit



In [8]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
    ])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(5)

# Tunnel sans compte
tunnel = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no",
     "-R", "80:localhost:8501",
     "localhost.run"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(5)
for line in tunnel.stdout:
    line = line.decode()
    if "https://" in line:
        print("✅ URL publique :", line.strip())
        break

✅ URL publique : Follow your favourite reverse tunnel at [https://twitter.com/localhost_run].
